### Indicators
- Revenue Growth (QoQ)
- EPS - Earnings Per Share
- Gross Margin - (Revenue - Cost of Goods) / Revenue
- Net Profit Margin - Percentage of revenue left after expenses
- ROE - Return on Equity, How effectively capital is being used
- ROIC - Return on Invested Capital, how well it turns all capital into profit
- ROA - Return on Assets
- Equity Multiplier (Total Assets / Stockholder's Equity)
- OCF - Operating cash flow
- CapEx - Expenditures
- FCF - (OCF - CapEx) Free Cash Flow, hard cash
- Cash King Ratio - (OCF / Net Income)
- D/E - Debt to Equity 
- EBIT (Earnings Before Interest & Taxes)
- EBITDA - EBIT + Deprecation and Ammortization (Cash generated by operations)
- Net Debt / EBITDA
- EV - Enterprise Value, EV = Market Cap + Total Debt - Cash and Equivalents
- EV/EBITDA 
- P/E - Price / Earnings
- PEG Ration (P/E) / (EPS growth rate)
- P/B - Price-to-Book, compares market value and assets
- EV/Revenue
- Current Ration (Current Assets / Current Liabilities)
- Net Debt (Total Debt - Cash) if we have more cash than debt
- Dividend Yield  (Annual dividend / Price)
- Dividend Payout Ratio: (Dividends / Net Income)
- Altman Z-Score (Z = 1.2A + 1.4B + 3.3C + 0.6D + 1.0E): 
    - A = Working Capital / Total Assets (Liquidity)
    - B = Retained Earnings / Total Assets (Accumulated Profitability)
    - C = EBIT / Total Assets (Current Profitability)
    - D = Market Value of Equity / Total Liabilities (Solvency)
    - E = Sales (Revenue) / Total Assets (Asset Efficiency)
- Sloan Ratio (Net Income - OCF) / Total Assets
- Owner Earnings (Net Income + Depreciation/Amortization – Maintenance CapEx)
- Rule of 40 (Revenue Growth % + FCF Margin %) (FCF Margin % = FCF / Revenue * 100)


In [1]:
import os
import requests
import json
from tqdm.notebook import tqdm
from datetime import date
import time

from edgar import *
import yfinance as yf
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
IDENTITY = os.getenv("IDENTITY")

COMPANY_TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
SEC_HEADERS = {
    "User-Agent": f"financial-research-masters-degree ({IDENTITY})",
    "Accept": "application/json"
}

EDGAR_DATA = "edgar_data"

MAX_BACKFILL_DAYS = 365 * 3

set_identity(IDENTITY)

RENAME_TABLE = {
    'revenue': 'totalRevenue',
    'operating_income': 'operatingIncome',
    'net_income': 'netIncome',
    'total_assets': 'totalAssets',
    'total_liabilities': 'totalLiabilities',
    'stockholders_equity': 'totalShareholderEquity',
    'current_assets': 'totalCurrentAssets',
    'current_liabilities': 'totalCurrentLiabilities',
    'operating_cash_flow': 'operatingCashflow',
    'capital_expenditures': 'capitalExpenditures',
    'free_cash_flow': 'freeCashFlow',
    'shares_outstanding_basic': 'commonStockSharesOutstanding',
    'shares_outstanding_diluted': 'commonStockSharesOutstandingDiluted'
}

METRICS = {
    'totalRevenue': [
        "Revenues", 
        "SalesRevenueNet", 
        "SalesRevenueGoodsNet", 
        "RevenuesNetOfInterestExpense",
    ],
    'operatingIncome': [
        "OperatingIncomeLoss",
        "OperatingIncomeLossFromContinuingOperations",
        "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest",
        "IncomeLossFromContinuingOperationsBeforeInterestExpenseInterestIncomeIncomeTaxesExtraordinaryItemsNoncontrollingInterestsNet",
        "IncomeLossFromContinuingOperationsBeforeIncomeTaxesMinorityInterestAndIncomeLossFromEquityMethodInvestments",
    ],
    'operatingExpenses': ["OperatingExpenses", "CostsAndExpenses", "LeaseCost"],
    'nonoperatingIncome': [
        "NonoperatingIncomeExpense",
        "IncomeLossFromNonoperatingActivities",
        "OtherNonoperatingIncomeExpense",
        "OtherNonoperatingIncome",
    ],
    'netIncome': [
        "NetIncomeLoss",
        "ProfitLoss",
        "NetIncomeLossAvailableToCommonStockholdersBasic",
        "IncomeLossFromContinuingOperations",
    ],
    'totalAssets': ["Assets"],
    'totalLiabilities': ["Liabilities"],
    'totalShareholderEquity': ["StockholdersEquity"],
    'totalCurrentAssets': ["AssetsCurrent"],
    'totalNonCurrentAssets': ["NoncurrentAssets"],
    'totalCurrentLiabilities': ["LiabilitiesCurrent"],
    'totalNonCurrentLiabilities': ["LiabilitiesNoncurrent"],
    'liabilitiesAndStockholdersEquity': ["LiabilitiesAndStockholdersEquity"],
    'operatingCashflow': [
        "NetCashProvidedByUsedInOperatingActivities",
        "NetCashProvidedByUsedInContinuingOperations",
        "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations",
        ],
    'capitalExpenditures': [
        "PaymentsToAcquirePropertyPlantAndEquipment", 
        "CapitalExpendituresIncurredButNotYetPaid",
        "PaymentsToAcquireProductiveAssets",
        "PaymentsForCapitalImprovements",
    ],
    
    'grossProfit': ["GrossProfit"],
    'costofGoodsAndServicesSold': [
        "CostOfGoodsAndServicesSold",
        "CostofGoodsAndServicesSold",
        # "CostsAndExpenses",
        "CostOfGoodsSold", 
        "CostOfRevenue", 
        "CostOfSales", 
        "CostOfServices",
        "CostOfPurchasedOilAndGas",
    ],
    'depreciation': ["Depreciation"],
    'amortization': ["AmortizationOfIntangibleAssets", "FinanceLeaseRightOfUseAssetAmortization"],
    'depreciationAndAmortization': [
        "DepreciationDepletionAndAmortization", 
        "DepreciationAndAmortization",
        "DepreciationAmortizationAndAccretionNet",
        "DepreciationDepletionAndAmortizationIncludingDiscontinuedOperationDepreciationandAmortization",
        "DepreciationDepletionAndAmortizationincludingdiscontinuedoperations",
        "DepreciationAndAmortizationIncludingDiscontinuedOperations"
        ],
    # 'costOfGoodsSoldDepreciationAndAmortization': ["CostOfGoodsSoldDepreciationAndAmortization"],
    'cashAndCashEquivalentsAtCarryingValue': [
        "cashAndCashEquivalentsAtCarryingValue", 
        "CashAndCashEquivalentsAtCarryingValue",
        "CashCashEquivalentsAndShortTermInvestments", 
        "CashAndCashEquivalentsPeriodIncreaseDecrease", 
        "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents",
        "CashAndDueFromBanks",
    ],
    'longTermInvestments': [ #TODO: Make use of it
        "LongTermInvestments",
        "AvailableForSaleSecuritiesNoncurrent",
        "InvestmentsAndAdvancesSecurities",
        "OtherInvestmentsNoncurrent"
    ],
    'shortTermDebt': [
        "DebtCurrent",                          # The "Holy Grail" - total current debt
        "ShortTermBorrowings",                  # Very common consolidated tag
        "LongTermDebtCurrent",                  # The portion of big loans due THIS year
        "LongTermDebtAndCapitalLeaseObligationsCurrent",
        "CommercialPaper",                      # Vital for Banks/Blue Chips
        "NotesPayableCurrent",                  # Standard for smaller loans
        "LinesOfCreditCurrent",                 # Revolver draws
        "ConvertibleDebtCurrent",               # Tech/Biotech specific
        "NotesAndLoansPayableCurrent",          # General bank debt
        "OtherShortTermBorrowings",             # The catch-all
        "SecuredDebtCurrent",                   # Specific type
        "UnsecuredShortTermBorrowings"          # Common for financials
    ],
    'longTermDebt': [
        "LongTermDebt", 
        "LongTermDebtAndCapitalLeaseObligations", 
        "LongTermDebtNoncurrent",
        "LongTermNotesPayable",
        "ConvertibleLongTermNotesPayable",
        "SeniorLongTermNotes",
        "UnsecuredLongTermDebt"
        ],
    'retainedEarnings': ["RetainedEarningsAccumulatedDeficit"],
    'dividendPayout': [
        "PaymentsOfDividends", 
        "PaymentsOfDividendsCommonStock",
        "DividendsCommonStockCash",
        "PaymentsOfOrdinaryDividends",
    ],
    'commonStockDividendsPerShareCashPaid': ['CommonStockDividendsPerShareCashPaid'],
    'interestExpense': [
        "InterestExpense",                           # Total (Best)
        "InterestAndDebtExpense",                    # Combined
        "InterestExpenseNonoperating",               # Specific to non-bank debt
        "InterestExpenseOperating",                  # Specific to banks/lending
        "InterestAndFeeExpensesOperating",           # Common for GS
    ],
    'incomeTaxExpense': [
        "IncomeTaxExpenseBenefit", 
    ],
    'currentIncomeTaxExpense': [
        "CurrentIncomeTaxExpenseBenefit",
    ],
    'deferredIncomeTaxExpense': [
        "DeferredIncomeTaxExpenseBenefit",
    ],
    'ebitda': [  # some companies report directly
        "EarningsBeforeInterestTaxesDepreciationAndAmortization",
    ],
    'researchAndDevelopment': [
        "ResearchAndDevelopmentExpense",
        "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost",
        "ResearchAndDevelopmentExpenseSoftwareExcludingAcquiredInProcessCost",
        "ResearchAndDevelopmentAssetAcquiredOtherThanThroughBusinessCombinationWrittenOff",
    ],
    'sellingGeneralAndAdministrative': [
        "SellingGeneralAndAdministrativeExpense",
        "GeneralAndAdministrativeExpense",
        "SellingAndMarketingExpense",
    ],
    'stockBasedCompensation': [
        "ShareBasedCompensation",
        "AllocatedShareBasedCompensationExpense",
        "EmployeeBenefitsAndShareBasedCompensation",
    ],
    'goodwill': [
        "Goodwill",
    ],
    'intangibleAssets': [
        "IntangibleAssetsNetExcludingGoodwill",
        "FiniteLivedIntangibleAssetsNet",
        "IndefiniteLivedIntangibleAssetsExcludingGoodwill",
    ],
    'currentNetReceivables': [
        "AccountsReceivableNetCurrent",
        "ReceivablesNetCurrent",
        "AccountsAndNotesReceivableNet",
        "AccountsReceivableNet",
        "IncreaseDecreaseInAccountsReceivable",
        "IncreaseDecreaseInReceivables",
    ],
    'inventory': [
        "InventoryNet",
        "InventoryGross",
        "FIFOInventoryAmount",
        "RetailRelatedInventory",
    ],
    'currentAccountsPayable': [
        "AccountsPayableCurrent",
        "AccountsPayableAndAccruedLiabilitiesCurrent",
        "IncreaseDecreaseInAccountsPayable",
    ],
    'accruedLiabilities': [
        "AccruedLiabilitiesCurrent",
        "EmployeeRelatedLiabilitiesCurrent",
        "OtherLiabilitiesCurrent",
    ],
    'deferredRevenue': [
        "DeferredRevenueCurrent",
        "DeferredRevenueNoncurrent",
        "ContractWithCustomerLiabilityCurrent",
        "ContractWithCustomerLiabilityNoncurrent",
    ],
    'propertyPlantEquipment': [
        "PropertyPlantAndEquipmentNet",
        "PropertyPlantAndEquipmentAndFinanceLeaseRightOfUseAssetAfterAccumulatedDepreciationAndAmortization",
    ],
    'commonStockSharesOutstanding': [
        "WeightedAverageNumberOfSharesOutstandingBasic",
        "CommonStockSharesOutstanding",
        "EntityCommonStockSharesOutstanding",
        "SharesOutstanding",
    ],
    'commonStockSharesOutstandingDiluted': [
        "WeightedAverageNumberOfDilutedSharesOutstanding",
        "WeightedAverageNumberOfSharesOutstandingDiluted",
        "WeightedAverageNumberOfShareOutstandingDiluted",
        ],
    'commonStockSharesOutstandingBasicAndDiluted': ["WeightedAverageNumberOfShareOutstandingBasicAndDiluted"],
    'proceedsFromRepurchaseOfEquity': [
        "PaymentsForRepurchaseOfCommonStock",
        "StockRepurchasedAndRetiredDuringPeriodValue",
        "TreasuryStockValueAcquiredCostMethod",
        "StockRepurchasedDuringPeriodValue",
    ],
    'operatingLeaseRightOfUseAsset': [
        "OperatingLeaseRightOfUseAsset",
        "FinanceLeaseRightOfUseAsset",
    ],
    'operatingLeaseLiability': [
        "OperatingLeaseLiability",
        "OperatingLeaseLiabilityCurrent",
        "OperatingLeaseLiabilityNoncurrent",
    ],
    'noncontrollingInterest': [
        "MinorityInterest",
        "MinorityInterestInSubsidiaries",
    ],
    'comprehensiveIncome': [
        "ComprehensiveIncomeNetOfTax",
        "OtherComprehensiveIncomeLossNetOfTax",
    ],
    # Financing flows — useful for FCF yield, leverage analysis
    'proceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet': [
        "ProceedsFromIssuanceOfLongTermDebt",
        "ProceedsFromIssuanceOfDebt",
        "ProceedsFromLinesOfCredit",
        "ProceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet",
        "ProceedsFromLoans",
        "ProceedsFromRepaymentsOfShortTermDebtMaturingInMoreThanThreeMonths_Duration",
    ],
    'repaymentsOfDebt': [
        "RepaymentsOfDebt",
        "RepaymentsOfLongTermDebt",
        "RepaymentsOfLinesOfCredit",
        "RepaymentsOfLongTermDebtAndCapitalSecurities",
        "RepaymentsOfCommercialPaper",
    ],
    'proceedsFromEquityIssuance': [
        "ProceedsFromIssuanceOfCommonStock",
        "ProceedsFromStockOptionsExercised",
    ],
}

In [ ]:
from util import *

def get_precise_quarterly_value(df, concept_tag, report_date):
    report_date_ts = pd.to_datetime(report_date)

    base = df[
        (df['clean_concept'] == concept_tag) &
        (df['dimension'].isna())
    ]
    if base.empty:
        return None, None

    # Instant values (balance sheet)
    # instants = base[base['period_start'].isna()]
    instants = base[
        (base['period_start'].isna()) &
        (base['period_end_ts'].notna()) &
        ((base['period_end_ts'] - report_date_ts).dt.days.abs() <= 7)
    ]
    if not instants.empty:
        return instants.iloc[0]['value'], instants.iloc[0]['source_filing_date']

    date_diff = (base['period_end_ts'] - report_date_ts).dt.days.abs()
    matches = base[date_diff <= 7].copy()
    if matches.empty:
        return None, None

    quarterly = matches[(matches['days'] > 80) & (matches['days'] < 115)]
    if not quarterly.empty:
        quarterly = quarterly.copy()
        quarterly['dist'] = (quarterly['days'] - 91).abs()
        val = quarterly.sort_values('dist').iloc[0]
        return val['value'], val['source_filing_date']

    return None, None


def get_precise_annual_value(df, concept_tag, report_date):
    report_date_ts = pd.to_datetime(report_date)

    base = df[
        (df['clean_concept'] == concept_tag) &
        (df['dimension'].isna())
    ]
    if base.empty:
        return None, None

    # instants = base[base['period_start'].isna()]
    instants = base[
        (base['period_start'].isna()) &
        (base['period_end_ts'].notna()) &
        ((base['period_end_ts'] - report_date_ts).dt.days.abs() <= 7)
    ]
    if not instants.empty:
        return instants.iloc[0]['value'], instants.iloc[0]['source_filing_date']

    date_diff = (base['period_end_ts'] - report_date_ts).dt.days.abs()
    matches = base[date_diff <= 7].copy()
    if matches.empty:
        return None, None

    annual = matches[(matches['days'] > 350) & (matches['days'] < 380)]
    if not annual.empty:
        annual = annual.copy()
        annual['dist'] = (annual['days'] - 365).abs()
        val = annual.sort_values('dist').iloc[0]
        return val['value'], val['source_filing_date']

    return None, None

def add_all_quarterly_metrics_from_df(facts_df: pd.DataFrame, report_date, metrics: dict):
    for my_name, gaap_tags in METRICS.items():
        if metrics.get(my_name) is not None:
            continue
        for tag in gaap_tags:
            val, _ = get_precise_quarterly_value(facts_df, tag, report_date)
            if val is not None:
                metrics[my_name] = val
                break
        # else:
        #     if metrics.get(my_name) is None:
        #         print(f"Pass 1 miss: {my_name} on {report_date}")  # temporary debug

def add_all_annual_metrics_from_df(facts_df: pd.DataFrame, report_date, metrics: dict):
    for my_name, gaap_tags in METRICS.items():
        if metrics.get(my_name) is not None:
            continue
        for tag in gaap_tags:
            val, _ = get_precise_annual_value(facts_df, tag, report_date)
            if val is not None:
                metrics[my_name] = val
                break

def parse_tenq_raw(filing, price_history: pd.Series, facts_df: pd.DataFrame) -> dict:
    tenq = filing.obj()
    raw_metrics = {}

    try:
        for old_name, value in tenq.financials.get_financial_metrics().items():
            if old_name in RENAME_TABLE:
                raw_metrics[RENAME_TABLE[old_name]] = value
    except Exception as e:
        print(f"Warning: financials extraction failed for 10-Q {filing.filing_date}: {e}")

    add_all_quarterly_metrics_from_df(facts_df, filing.period_of_report, raw_metrics)

    raw_metrics['fiscalDateEnding'] = tenq.period_of_report
    raw_metrics['filing_date'] = tenq.filing_date
    raw_metrics['stock_price'] = lookup_price(price_history, tenq.filing_date)
    raw_metrics['currency_symbol'] = tenq.financials.get_currency_symbol()
    raw_metrics['form_type'] = '10-Q'

    if raw_metrics.get('commonStockSharesOutstanding') is None:
        raw_metrics['commonStockSharesOutstanding'] = get_shares_from_cover_page_from_df(facts_df)

    return sanitize_metrics(raw_metrics)


def extract_annual_metrics(filing, price_history: pd.Series, facts_df: pd.DataFrame) -> dict:
    tenk = filing.obj()
    raw_metrics = {}

    try:
        for old_name, value in tenk.financials.get_financial_metrics().items():
            if old_name in RENAME_TABLE:
                raw_metrics[RENAME_TABLE[old_name]] = value
    except Exception as e:
        print(f"Warning: financials extraction failed for 10-K {filing.filing_date}: {e}")

    add_all_annual_metrics_from_df(facts_df, filing.period_of_report, raw_metrics)

    raw_metrics['fiscalDateEnding'] = tenk.period_of_report
    raw_metrics['filing_date'] = filing.filing_date
    raw_metrics['stock_price'] = lookup_price(price_history, filing.filing_date)
    raw_metrics['currency_symbol'] = tenk.financials.get_currency_symbol()

    if raw_metrics.get('commonStockSharesOutstanding') is None:
        raw_metrics['commonStockSharesOutstanding'] = get_shares_from_cover_page_from_df(facts_df)

    return sanitize_metrics(raw_metrics)


def get_shares_from_cover_page_from_df(facts_df: pd.DataFrame) -> float | None:
    """Same as get_shares_from_cover_page but reuses already-parsed facts_df."""
    try:
        cover = facts_df[
            (facts_df['clean_concept'] == 'EntityCommonStockSharesOutstanding') &
            (facts_df['period_start'].isna()) &
            (facts_df['dimension'].isna())
        ]
        if not cover.empty:
            return float(cover.iloc[-1]['value'])
    except Exception as e:
        print(f"Cover page shares lookup failed: {e}")
    return None

def prepare_facts_df(facts_df: pd.DataFrame) -> pd.DataFrame:
    """Pre-compute all derived columns once so lookups are just filters."""
    facts_df = facts_df.copy()
    if 'dimension' not in facts_df.columns:
        facts_df['dimension'] = None
    facts_df['clean_concept'] = facts_df['concept'].str.split(':|_').str[-1]

    # Normalise: use period_instant as period_end when period_end is missing
    if 'period_instant' in facts_df.columns:
        facts_df['period_end'] = facts_df['period_end'].fillna(facts_df['period_instant'])

    facts_df['period_end_ts'] = pd.to_datetime(facts_df['period_end'])
    facts_df['period_start_ts'] = pd.to_datetime(facts_df['period_start'])
    facts_df['days'] = (facts_df['period_end_ts'] - facts_df['period_start_ts']).dt.days
    return facts_df

FLOW_METRICS = {
    'totalRevenue', 'netIncome', 'grossProfit', 'operatingIncome',
    'operatingCashflow', 'capitalExpenditures', 'incomeTaxExpense',
    'interestExpense', 'depreciationAndAmortization', 'stockBasedCompensation',
    'operatingExpenses', 'nonoperatingIncome', 'researchAndDevelopment',
    'sellingGeneralAndAdministrative', 'comprehensiveIncome', 'freeCashFlow',
    'depreciation', 'amortization', 'dividendPayout', 'proceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet', 
    'repaymentsOfDebt', 'proceedsFromEquityIssuance', 'proceedsFromRepurchaseOfEquity',
    'costofGoodsAndServicesSold', 'ebitda', 'currentIncomeTaxExpense', 
    'deferredIncomeTaxExpense'
}
STOCK_METRICS = {
    'totalAssets', 'totalLiabilities', 'totalShareholderEquity',
    'totalCurrentAssets', 'totalCurrentLiabilities', 'longTermDebt',
    'shortTermDebt', 'retainedEarnings', 'cashAndCashEquivalentsAtCarryingValue',
    'currentNetReceivables', 'inventory', 'propertyPlantEquipment', 'goodwill', 
    'intangibleAssets', 'operatingLeaseRightOfUseAsset', 'liabilitiesAndStockholdersEquity',
    'currentAccountsPayable', 'accruedLiabilities', 'operatingLeaseLiability', 
    'noncontrollingInterest', 'deferredRevenue', 'longTermInvestments', 
    'totalNonCurrentAssets', 'totalNonCurrentLiabilities'
}
SKIP_METRICS = {
    # These are weighted averages or per-share — don't subtract
    'commonStockSharesOutstanding',
    'commonStockSharesOutstandingDiluted',
    'commonStockDividendsPerShareCashPaid',
}

def derive_q4_metrics(annual_metrics: dict, q1: dict, q2: dict, q3: dict) -> dict:
    q4 = {}
    for metric in FLOW_METRICS:
        annual = annual_metrics.get(metric)
        prior = sum(filter(None, [q1.get(metric), q2.get(metric), q3.get(metric)]))
        q4[metric] = (annual - prior) if annual is not None else None

    for metric in STOCK_METRICS:
        q4[metric] = annual_metrics.get(metric) 

    for metric in SKIP_METRICS:
        q4[metric] = annual_metrics.get(metric)

    return q4
    
def extract_all_facts(filing, form_type: str) -> pd.DataFrame:
    xbrl = filing.xbrl()
    facts_df = xbrl.facts.to_dataframe()
    facts_df['source_filing_date'] = filing.filing_date
    facts_df['source_form_type'] = form_type
    return prepare_facts_df(facts_df)

def parse_full_history(ticker: str) -> pd.DataFrame | None:
    company = Company(ticker)
    price_history = get_price_history(ticker)
    
    filings_10q = [
        ("10-Q", f) for f in company.get_filings(form="10-Q", amendments=False)
        if f.filing_date >= date(2009, 1, 1)
    ]
    filings_10k = [
        ("10-K", f) for f in company.get_filings(form="10-K", amendments=False)
        if f.filing_date >= date(2009, 1, 1)
    ]

    if not filings_10q:
        print(f"Warning: No 10-Q filings found for {ticker}")
        return None
    
    all_filings = sorted(filings_10q + filings_10k, key=lambda x: x[1].period_of_report)

    # ── PASS 1: raw extraction only ──────────────────────────────────────────
    rows = []
    all_facts = []
    quarterly_buffer = []

    for form_type, filing in tqdm(all_filings, desc="Pass 1: extracting raw metrics"):
        try:
            facts_df = extract_all_facts(filing, form_type)
            all_facts.append(facts_df)

            if form_type == "10-Q":
                raw = parse_tenq_raw(filing, price_history, facts_df)
                rows.append(raw)
                quarterly_buffer.append(raw)

            elif form_type == "10-K":
                if len(quarterly_buffer) < 3:
                    print(f"Warning: Fewer than 3 quarters buffered before 10-K {filing.filing_date}, skipping Q4 derivation")
                    quarterly_buffer = []
                    continue

                annual_metrics = extract_annual_metrics(filing, price_history, facts_df)
                q1, q2, q3 = quarterly_buffer[-3], quarterly_buffer[-2], quarterly_buffer[-1]
                q4_raw = derive_q4_metrics(annual_metrics, q1, q2, q3)

                q4_raw['fiscalDateEnding'] = annual_metrics['fiscalDateEnding']
                q4_raw['filing_date'] = annual_metrics['filing_date']
                q4_raw['stock_price'] = annual_metrics['stock_price']
                q4_raw['currency_symbol'] = annual_metrics['currency_symbol']
                q4_raw['form_type'] = '10-K-derived'

                rows.append(q4_raw)
                quarterly_buffer = []
                
        except Exception as e:
                print(f"Skipping {form_type} filing {filing.filing_date}: {e}")
                continue
    
    if not rows:
        print(f"Warning: No data found for {ticker}")
        return None
    
    # ── PASS 2: backfill missing values from global facts pool ───────────────
    print("Pass 2: backfilling from later filings...")
    global_facts = pd.concat(all_facts, ignore_index=True)
    global_facts = prepare_facts_df(global_facts) 
    del all_facts

    global_facts_by_concept = {
        concept: group 
        for concept, group in global_facts.groupby('clean_concept')
    }

    for row in tqdm(rows, desc="Pass 2: backfilling"):
        report_date = str(row.get('fiscalDateEnding', ''))[:10]
        if not report_date:
            print("Warning: Missing report date, skipping")
            continue

        is_annual = row.get('form_type') == '10-K-derived'

        for my_name, gaap_tags in METRICS.items():
            if row.get(my_name) is not None:
                continue
            for tag in gaap_tags:
                tag_df = global_facts_by_concept.get(tag)
                if tag_df is None:
                    continue

                val, end_date_where_found = (
                    get_precise_annual_value(tag_df, tag, report_date)
                    if is_annual else
                    get_precise_quarterly_value(tag_df, tag, report_date)
                )
                if val is not None:
                    # filing_date = pd.to_datetime(end_date_where_found)
                    # period_date = pd.to_datetime(report_date)
                    # days_gap = (filing_date - period_date).days
                    # if days_gap > MAX_BACKFILL_DAYS:
                    #     continue
                    # print(f"Found tag: {my_name} for date {report_date} in statement reported at: {end_date_where_found}")
                    row[my_name] = sanitize_numeric(val)
                    break
        if row.get('operatingCashflow') is None:
            print(f"Missing operatingCashflow in {report_date}")

    del global_facts
    del global_facts_by_concept

    # ── PASS 3: accounting logic + indicators, once, in order ────────────────
    print("Pass 3: computing accounting logic and indicators...")

    prev_raw = None
    for row in tqdm(rows, desc="Pass 3: computing indicators"):
        apply_accounting_logic(row)

        indicators = compute_indicators(row, prev_raw, 'quarterly')
        for k, v in indicators.items():
            row[f"ind_{k}"] = v
        
        prev_raw = {k: v for k, v in row.items() if not k.startswith('ind_')}

    # ── Build final DataFrame ─────────────────────────────────────────────────
    df = pd.DataFrame(rows)
    del rows
    df['form_type'] = df['form_type'].fillna('10-Q')
    df.set_index("fiscalDateEnding", inplace=True)
    df['filing_date'] = pd.to_datetime(df['filing_date'])
    df.index = pd.to_datetime(df.index)

    return df

def save_tenq_history(ticker: str, df: pd.DataFrame):
    path = f"{EDGAR_DATA}/{ticker}.csv"
    os.makedirs(EDGAR_DATA, exist_ok=True)

    df_to_save = df.reset_index() if df.index.name else df
    df_to_save.to_csv(path, index=False)
    print(f"Success: Overwrote history for {ticker} at {path}")
    


TICKER = "UNH"


def data_pipeline():
    tickers_data = get_company_tickers()

    for entry in tickers_data.values():
        # ticker = entry["ticker"]
        start_time = time.time()
        ticker = TICKER
        print(f"Parsing data for {entry["title"]} ({ticker})")

        metadata = get_company_metadata(ticker)
        if metadata["sector"] is None:
            print(f"Warning: Could not derive operating sector of: {entry["title"]} ({ticker}), skipping")
            continue

        metadata["title"] = entry["title"]
        save_metadata(ticker, metadata, f"{EDGAR_DATA}/metadata.json")
        try:
            df = parse_full_history(ticker)
        except Exception as e:
            print(f"Error during parsing filings, skipping: {str(e)}")
            continue

        if df is not None:
            save_tenq_history(ticker, df)
            print(f"Successfully finished parsing {entry["title"]} ({ticker}) data, rows: {len(df)}")
            print(f"Total time: {round(time.time() - start_time, 4)}s")
        else:
            print(f"DataFrame for {ticker} is None, skipping")
            continue

        
        break

def export_for_training() -> pd.DataFrame:
    with open(f"{EDGAR_DATA}/metadata.json", "r") as f:
        metadata = json.load(f)

    frames = []
    for ticker, meta in metadata.items():
        path = f"{EDGAR_DATA}/{ticker}.csv"
        if os.path.exists(path):
            base = pd.read_csv(path)

            extra = pd.DataFrame({
                "ticker": ticker,
                "sector": meta.get("sector"),
            }, index=base.index)

            df = pd.concat([base, extra], axis=1)
            frames.append(df)

    if frames:
        return pd.concat(frames, ignore_index=True)
    else:
        print("Warnign: No frames found, returning empty DF")
        return pd.DataFrame()

data_pipeline()

Parsing data for NVIDIA CORP (UNH)


Pass 1: extracting raw metrics:   0%|          | 0/69 [00:00<?, ?it/s]

No XBRL attachments found in filing Filing(company='UNITEDHEALTH GROUP INC', cik=731766, form='10-K', filing_date='2009-02-11', accession_no='0001193125-09-025587')
No XBRL attachments found in filing Filing(company='UNITEDHEALTH GROUP INC', cik=731766, form='10-Q', filing_date='2009-05-07', accession_no='0001193125-09-103335')


Skipping 10-K filing 2009-02-11: 'NoneType' object has no attribute 'facts'
Skipping 10-Q filing 2009-05-07: 'NoneType' object has no attribute 'facts'
Pass 2: backfilling from later filings...


Pass 2: backfilling:   0%|          | 0/66 [00:00<?, ?it/s]

Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30
Missing operatingCashflow in 2009-06-30


Pass 3: computing indicators:   0%|          | 0/66 [00:00<?, ?it/s]

Success: Overwrote history for UNH at edgar_data/UNH.csv
Successfully finished parsing NVIDIA CORP (UNH) data, rows: 66
Total time: 110.5745s


In [11]:
pd.set_option('display.max_rows', None)
PATH = f"edgar_data/{TICKER}.csv"

df = pd.read_csv(PATH)
print(len(df))
# df[df['fiscal_date_ending'] == '2022-09-30'].head()
df.isna().sum()

# df = export_for_training()
# df = df[df['ticker'] == TICKER]
# df.tail()
# df.isna().sum()

66


fiscalDateEnding                                              0
totalRevenue                                                  0
operatingIncome                                               0
netIncome                                                     0
totalAssets                                                   0
totalLiabilities                                              0
totalShareholderEquity                                        0
totalCurrentAssets                                            0
totalCurrentLiabilities                                       0
operatingCashflow                                            34
capitalExpenditures                                           0
freeCashFlow                                                 34
commonStockSharesOutstanding                                  0
commonStockSharesOutstandingDiluted                           0
operatingExpenses                                             0
liabilitiesAndStockholdersEquity        

In [14]:

ticker = TICKER

company = Company(ticker)
filings = company.get_filings(form="10-Q")

filing = None
for f in filings:
    # print(f.period_of_report)
    if f.period_of_report == "2025-09-30":
        filing = f
        break
    
# filing = filings[10]
xbrl = filing.xbrl()

facts = xbrl.facts.to_dataframe().to_json('out.json', indent=2)



TODO:
- List of key fundamentals (ones that are actually used to count metrics)
- Filter out data points that are missing a lot of those key fundamentals
- List of values for which it is valid to use interpolation to fill missing value
- At the end of pipeline, if big percentage of indicators are missing, get rid of this data (company) completely
- How to deal with missing quarters (explicitly removed) when training models in sliding windows (~5 training), predict whether price change will beat markete change during next year?
- Collect dataframe with whole market price (S&P ?) for years 2009 - Present (for comparison)